# Generic search: what changed in the maps

The defaults of the generic search changed on 2026-11-18: template-fit VIS instead of PSF, and the match filter of the
optimization on Aliqa Uma (`notebooks/optimization/stream_optimization.ipynb`): Dotter2016 instead of Marigo2017,
`error_multiplier` 3 instead of 2, `red_cap_value` 0.55 instead of 0.75. Four selections, one change at a time, on the
same stars (morphology cuts, `POINT_LIKE_PROB` ≥ 0.5, the Planck dust correction for all):
- **yesterday**: Marigo2017, PSF VIS, the streamobs width and red cap;
- **templfit only**: the same with the template-fit VIS;
- **match filter only**: the new match filter (Dotter2016, `error_multiplier` 3, `red_cap_value` 0.55) with the PSF VIS;
- **both (current defaults)**: `DEFAULT_ISOCHRONE_MODEL` and `DEFAULT_MATCH_FILTER`.

Section 2: their corrected maps (`residual`: density − background, stars per pixel), on one colour scale. Section 3:
yesterday and today at matched distances (the template-fit VIS puts a stream about 0.2 mag farther). The SNR of the
candidates against the distance modulus, per selection: `notebooks/candidate_snr_comparison.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.analysis.generic_stream_search import generic_search_single_dm, prepare_generic_search
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.survey_footprint import survey_footprint_healpix_mask
from streameucliddes.plotting.plot_generic_search import DENSITY_PERCENTILES, plot_generic_search_residual_map
from streameucliddes.unit_conversions import get_isochrone_model

## 1. Stars, footprint and the four searches

The stars are read once, with the template-fit VIS (`vis_photometry="templfit"`, the default: the PSF VIS is read
too), and shared by the searches (none copies the catalog).

In [ ]:
CATALOG = "euclid_dr1"
SUBSET = "south"  # "south", "northwest", "northeast" or None
NSIDE = 512  # pixels of about 0.11 deg
SMOOTHING_DEG = 0.3
DISTANCE_MODULI = [16.0, 17.0, 18.0, 19.0]

# The selections: match filter and isochrone model (None: DEFAULT_ISOCHRONE_MODEL)
VIS_H = {"reference_band": ("euclid_dr1", "VIS"), "color_bands": [("euclid_dr1", "H")]}
NEW_FILTER = {key: get_match_filter()[key] for key in ("error_multiplier", "red_cap_value")}
SELECTIONS = {
    "yesterday": {"match_filter": {"name": "yesterday", **VIS_H, "vis_photometry": "psf"},
                  "isochrone_model": "Marigo2017"},
    "templfit only": {"match_filter": {"name": "templfit_only", **VIS_H, "vis_photometry": "templfit"},
                      "isochrone_model": "Marigo2017"},
    "match filter only": {"match_filter": {"name": "match_filter_only", **VIS_H, **NEW_FILTER, "vis_photometry": "psf"},
                          "isochrone_model": "Dotter2016"},
    "both (current defaults)": {"match_filter": get_match_filter(), "isochrone_model": None},
}
for name, selection in SELECTIONS.items():
    print(f"{name}: {get_isochrone_model(selection['isochrone_model'])}, {selection['match_filter']}")

In [ ]:
stars = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, "morphology_cuts"), subset=SUBSET)
footprint = survey_footprint_healpix_mask(stars["ra"].to_numpy(), stars["dec"].to_numpy(), NSIDE, nest=False)
print(f"{len(stars):,} stars; footprint {(~footprint).sum() * hp.nside2pixarea(NSIDE, degrees=True):,.0f} deg2")

searches = {name: prepare_generic_search(stars, match_filter=selection["match_filter"],
                                         isochrone_model=selection["isochrone_model"], nside=NSIDE,
                                         footprint_mask=footprint, nest=False)
            for name, selection in SELECTIONS.items()}
for name, search in searches.items():
    print(f"{name}: {search.keep.sum():,} stars pass the cuts; isochrone {search.isochrones[0]}")



def robust_std(values):
    """1.4826 x the median absolute deviation: the standard deviation of Gaussian values, robust to outliers."""
    return 1.4826 * np.median(np.abs(values - np.median(values)))

## 2. Corrected maps: one change at a time

The residual (density − background, stars per pixel) of each selection, on one colour scale per distance modulus
(the `DENSITY_PERCENTILES` of yesterday's map), with the number of stars selected and the spread of the residual
(1.4826 × the median absolute deviation, inside the footprint). The residual noise grows with the number of stars
selected.

In [ ]:
results = {name: {dm: generic_search_single_dm(distance_modulus=dm, search=search, smoothing_deg=SMOOTHING_DEG)["residual"]
                  for dm in DISTANCE_MODULI}
           for name, search in searches.items()}
template = generic_search_single_dm(distance_modulus=DISTANCE_MODULI[0], search=searches["yesterday"],
                                    smoothing_deg=SMOOTHING_DEG)  # footprint, nside... of the maps, for the plots

In [ ]:
for dm in DISTANCE_MODULI:
    vmin, vmax = np.percentile(np.ma.compressed(results["yesterday"][dm]), DENSITY_PERCENTILES)
    fig, axes = plt.subplots(2, 2, figsize=(16.0, 11.0), layout="constrained")
    for ax, name in zip(axes.ravel(), SELECTIONS):
        residual = results[name][dm]
        spread = robust_std(np.ma.compressed(residual))
        plot_generic_search_residual_map({**template, "residual": residual}, ax=ax, vmin=vmin, vmax=vmax,
                                         title=f"{name}: spread {spread:.3f} stars per pixel")
    fig.suptitle(f"Residual (stars per pixel), dm {dm:g}")
    plt.show()

## 3. Yesterday and today at matched distances

With the template-fit VIS, a stream is about 0.2 mag farther for the isochrone than with the PSF VIS. On Aliqa Uma,
whose distance is known (Li et al. 2021), the SNR peaks at the literature distance with the template-fit VIS and
0.15 to 0.2 mag closer with the PSF VIS (`notebooks/optimization/stream_optimization.ipynb`; the template-fit VIS is
0.07 mag brighter in DR1). Yesterday's map at dm X is thus to be compared with today's at X + `DM_SHIFT`.

In [ ]:
DM_SHIFT = 0.2
MATCHED_DMS = [16.8, 17.0, 17.2]  # yesterday's distance moduli; today's: + DM_SHIFT
TODAY = "both (current defaults)"

for dm in MATCHED_DMS:
    yesterday = generic_search_single_dm(distance_modulus=dm, search=searches["yesterday"], smoothing_deg=SMOOTHING_DEG)
    today = generic_search_single_dm(distance_modulus=dm + DM_SHIFT, search=searches[TODAY], smoothing_deg=SMOOTHING_DEG)
    vmin, vmax = np.percentile(np.ma.compressed(yesterday["residual"]), DENSITY_PERCENTILES)
    fig, axes = plt.subplots(1, 2, figsize=(16.0, 5.5), layout="constrained")
    for ax, (label, result) in zip(axes, ((f"yesterday, dm {dm:g}", yesterday),
                                          (f"today ({TODAY}), dm {dm + DM_SHIFT:g}", today))):
        plot_generic_search_residual_map(result, ax=ax, vmin=vmin, vmax=vmax, title=label)
    fig.suptitle("Residual (stars per pixel), at matched distances, yesterday's colour scale")
    plt.show()
    del yesterday, today